# Stock Price Prediction with LSTM and Attention

This notebook demonstrates the complete pipeline for stock price prediction using LSTM with attention mechanism.

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '../src')

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# For reproducibility
SEED = 42
np.random.seed(SEED)

## 1. Data Loading

Download historical stock data from Yahoo Finance.

In [ ]:
from data_loader import StockDataLoader

# Initialize loader
loader = StockDataLoader(cache_dir='../data')

# Download AAPL data
ticker = 'AAPL'
data = loader.download_stock_data(ticker, '2019-01-01', '2024-01-01')

print(f"Downloaded {len(data)} trading days of data")
print(f"Date range: {data.index.min()} to {data.index.max()}")
data.head()

In [ ]:
# Visualize price history
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Price
axes[0].plot(data.index, data['Close'], linewidth=1.5)
axes[0].set_title(f'{ticker} Stock Price', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(data.index, data['Volume'], width=1, alpha=0.7)
axes[1].set_title(f'{ticker} Trading Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Data summary
summary = loader.get_data_summary(data)
print("Data Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

## 2. Feature Engineering

Generate 15+ technical indicators for the model.

In [ ]:
from feature_engineering import FeatureEngineer

# Initialize engineer
engineer = FeatureEngineer()

# Generate all features
data_with_features = engineer.add_all_features(data)

print(f"Original columns: {len(data.columns)}")
print(f"Total columns after features: {len(data_with_features.columns)}")
print(f"\nGenerated {len(engineer.feature_columns)} features:")
for i, col in enumerate(engineer.feature_columns[:20]):
    print(f"  {i+1}. {col}")
if len(engineer.feature_columns) > 20:
    print(f"  ... and {len(engineer.feature_columns) - 20} more")

In [ ]:
# Visualize key indicators
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# Price with Moving Averages
ax = axes[0]
ax.plot(data_with_features.index, data_with_features['Close'], label='Close', linewidth=1.5)
ax.plot(data_with_features.index, data_with_features['SMA_20'], label='SMA 20', linewidth=1, alpha=0.8)
ax.plot(data_with_features.index, data_with_features['SMA_50'], label='SMA 50', linewidth=1, alpha=0.8)
ax.set_title('Price with Moving Averages', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# RSI
ax = axes[1]
ax.plot(data_with_features.index, data_with_features['RSI_14'], linewidth=1)
ax.axhline(y=70, color='r', linestyle='--', alpha=0.5)
ax.axhline(y=30, color='g', linestyle='--', alpha=0.5)
ax.fill_between(data_with_features.index, 30, 70, alpha=0.1)
ax.set_title('RSI (14)', fontsize=12, fontweight='bold')
ax.set_ylabel('RSI')
ax.grid(True, alpha=0.3)

# MACD
ax = axes[2]
ax.plot(data_with_features.index, data_with_features['MACD'], label='MACD', linewidth=1)
ax.plot(data_with_features.index, data_with_features['MACD_Signal'], label='Signal', linewidth=1)
ax.bar(data_with_features.index, data_with_features['MACD_Histogram'], alpha=0.5, width=1)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('MACD', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Bollinger Bands
ax = axes[3]
ax.plot(data_with_features.index, data_with_features['Close'], label='Close', linewidth=1)
ax.plot(data_with_features.index, data_with_features['BB_Upper'], label='Upper Band', linewidth=1, alpha=0.7)
ax.plot(data_with_features.index, data_with_features['BB_Lower'], label='Lower Band', linewidth=1, alpha=0.7)
ax.fill_between(data_with_features.index, data_with_features['BB_Lower'], 
                data_with_features['BB_Upper'], alpha=0.1)
ax.set_title('Bollinger Bands', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Handle missing values
data_clean = engineer.handle_missing_values(data_with_features, method='drop')
print(f"Clean data shape: {data_clean.shape}")
print(f"Removed {len(data_with_features) - len(data_clean)} rows with missing values")

In [ ]:
# Feature correlation heatmap
key_features = ['Close', 'Returns', 'RSI_14', 'MACD', 'BB_Width', 'ATR', 
                'Momentum_10', 'OBV', 'Volatility_20d', 'MFI']
key_features = [f for f in key_features if f in data_clean.columns]

plt.figure(figsize=(10, 8))
correlation = data_clean[key_features].corr()
sns.heatmap(correlation, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Sequence Building

Create sliding window sequences for LSTM input.

In [ ]:
from sequence_builder import SequenceBuilder, create_train_val_test_sequences

# Define feature columns
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Returns', 'Log_Returns',
    'SMA_20', 'SMA_50', 'EMA_20',
    'RSI_14', 'RSI_7',
    'MACD', 'MACD_Signal', 'MACD_Histogram',
    'BB_Upper', 'BB_Lower', 'BB_Width', 'BB_Percent',
    'ATR', 'Volatility_20d',
    'Momentum_10', 'Momentum_20',
    'OBV', 'VWAP',
    'Stoch_K', 'Williams_R', 'CCI',
    'ROC_10', 'MFI'
]
feature_cols = [c for c in feature_cols if c in data_clean.columns]
print(f"Using {len(feature_cols)} features")

# Initialize sequence builder
SEQUENCE_LENGTH = 60
FORECAST_HORIZON = 7

builder = SequenceBuilder(
    sequence_length=SEQUENCE_LENGTH,
    forecast_horizon=FORECAST_HORIZON,
    feature_columns=feature_cols,
    target_column='Close'
)

In [ ]:
# Create train/val/test sequences
(X_train, y_train), (X_val, y_val), (X_test, y_test), test_dates = \
    create_train_val_test_sequences(data_clean, builder, 0.7, 0.15, 0.15)

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Visualize a sample sequence
sample_idx = 0
sample_sequence = X_train[sample_idx]
sample_target = y_train[sample_idx]

# Unscale for visualization
close_idx = feature_cols.index('Close')

fig, ax = plt.subplots(figsize=(12, 5))

# Plot sequence (scaled)
ax.plot(range(SEQUENCE_LENGTH), sample_sequence[:, close_idx], 
        label='Input Sequence (Close)', linewidth=2)
ax.plot(range(SEQUENCE_LENGTH, SEQUENCE_LENGTH + FORECAST_HORIZON), 
        sample_target, label='Target (Next 7 Days)', 
        linewidth=2, linestyle='--', marker='o')
ax.axvline(x=SEQUENCE_LENGTH - 0.5, color='gray', linestyle=':', linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('Scaled Close Price')
ax.set_title('Sample Sequence: Input and Target', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. Model Building

Build LSTM model with attention mechanism.

In [ ]:
import tensorflow as tf
tf.random.set_seed(SEED)

from model import build_attention_model_with_weights, get_callbacks

# Build model
n_features = X_train.shape[2]
input_shape = (SEQUENCE_LENGTH, n_features)

model, attention_model = build_attention_model_with_weights(
    input_shape=input_shape,
    output_steps=FORECAST_HORIZON,
    lstm_units=[128, 64],
    attention_units=64,
    dropout_rate=0.2,
    learning_rate=0.001
)

model.summary()

## 5. Training

Train the model with early stopping and learning rate scheduling.

In [ ]:
# Training callbacks
callbacks = get_callbacks(
    model_path='../models/lstm_attention.keras',
    patience=15,
    reduce_lr_patience=5
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training and Validation MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation

Evaluate model performance on the test set.

In [ ]:
from evaluate import ModelEvaluator

# Get predictions with attention weights
predictions, attention_weights = attention_model.predict(X_test, verbose=0)

# Inverse transform to original scale
pred_unscaled = builder.inverse_transform_predictions(predictions)
actual_unscaled = builder.inverse_transform_predictions(y_test)

print(f"Predictions shape: {pred_unscaled.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator(results_dir='../results', ticker=ticker)

# Calculate metrics
metrics = evaluator.calculate_metrics(pred_unscaled, actual_unscaled)
print("\n=== Test Set Metrics ===")
for metric, value in metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

In [ ]:
# Per-horizon metrics
print("\n=== Metrics by Forecast Horizon ===")
for h in range(FORECAST_HORIZON):
    h_metrics = evaluator.calculate_metrics(pred_unscaled, actual_unscaled, horizon=h)
    print(f"Day {h+1}: RMSE=${h_metrics['rmse']:.2f}, MAE=${h_metrics['mae']:.2f}, MAPE={h_metrics['mape']:.2f}%")

In [ ]:
# Plot actual vs predicted
evaluator.plot_predictions(
    pred_unscaled, actual_unscaled, 
    dates=test_dates,
    save=True, show=True
)

In [ ]:
# Plot rolling RMSE
evaluator.plot_rolling_rmse(
    pred_unscaled, actual_unscaled,
    dates=test_dates,
    window=20,
    save=True, show=True
)

In [ ]:
# Plot horizon performance
evaluator.plot_horizon_performance(
    pred_unscaled, actual_unscaled,
    save=True, show=True
)

In [ ]:
# Plot attention weights
evaluator.plot_attention_weights(
    attention_weights,
    n_samples=5,
    save=True, show=True
)

In [ ]:
# Attention heatmap
evaluator.plot_attention_heatmap(
    attention_weights,
    n_samples=50,
    save=True, show=True
)

In [ ]:
# Error distribution analysis
evaluator.plot_error_distribution(
    pred_unscaled, actual_unscaled,
    save=True, show=True
)

## 7. Future Predictions

Make predictions for the next 7 days.

In [ ]:
from sequence_builder import create_single_prediction_sequence

# Get latest data point for prediction
latest_sequence = create_single_prediction_sequence(data_clean, builder)

# Predict
future_pred, future_attention = attention_model.predict(latest_sequence, verbose=0)
future_prices = builder.inverse_transform_predictions(future_pred)[0]

print("\n=== Next 7 Days Prediction ===")
print(f"Last known price: ${data_clean['Close'].iloc[-1]:.2f}")
print("\nPredicted prices:")
for i, price in enumerate(future_prices):
    change = (price - data_clean['Close'].iloc[-1]) / data_clean['Close'].iloc[-1] * 100
    print(f"  Day {i+1}: ${price:.2f} ({change:+.2f}%)")

In [ ]:
# Visualize prediction with recent history
recent_days = 60
recent_data = data_clean.iloc[-recent_days:]

fig, ax = plt.subplots(figsize=(14, 6))

# Historical
ax.plot(range(recent_days), recent_data['Close'].values, 
        label='Historical', linewidth=2, color='#2E86AB')

# Predictions
ax.plot(range(recent_days, recent_days + FORECAST_HORIZON), future_prices,
        label='Predicted', linewidth=2, linestyle='--', 
        marker='o', color='#E94F37', markersize=8)

ax.axvline(x=recent_days - 0.5, color='gray', linestyle=':', linewidth=2, label='Today')
ax.set_xlabel('Days')
ax.set_ylabel('Price ($)')
ax.set_title(f'{ticker} Price Prediction - Next 7 Days', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Save Model

Save the trained model for future use.

In [ ]:
import json
import os

# Save model
os.makedirs('../models', exist_ok=True)
model.save(f'../models/{ticker}_lstm_attention_final.keras')
print(f"Model saved to ../models/{ticker}_lstm_attention_final.keras")

# Save model parameters
params = {
    'ticker': ticker,
    'sequence_length': SEQUENCE_LENGTH,
    'forecast_horizon': FORECAST_HORIZON,
    'feature_columns': feature_cols,
    'final_metrics': metrics
}

with open(f'../models/{ticker}_params.json', 'w') as f:
    json.dump(params, f, indent=2)
print(f"Parameters saved to ../models/{ticker}_params.json")

## Summary

This notebook demonstrated:

1. **Data Loading**: Downloaded 5 years of AAPL stock data
2. **Feature Engineering**: Generated 15+ technical indicators
3. **Sequence Building**: Created sliding window sequences for LSTM
4. **Model Building**: Built LSTM with attention mechanism
5. **Training**: Trained with early stopping and LR scheduling
6. **Evaluation**: Comprehensive metrics and visualizations
7. **Prediction**: 7-day ahead forecasting

The attention mechanism allows the model to learn which historical time steps are most important for making predictions.